# Analyse Superstore

Ce notebook charge le jeu de données, le prétraite, puis répond aux questions avec des agrégations pandas et des visualisations.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Chargement et prétraitement minimal
df = pd.read_excel("US Superstore data.xls")
df = df.dropna(subset=["State", "City", "Customer Name", "Sales", "Profit"])
df.head()

## Quels sont les États qui enregistrent le plus de ventes ?

In [ ]:
sales_by_state = df.groupby('State')['Sales'].sum().sort_values(ascending=False)
print("Top 10 États par chiffre d'affaires :")
print(sales_by_state.head(10))

plt.figure(figsize=(10, 5))
sales_by_state.head(10).plot(kind='bar', color='steelblue')
plt.title("Top 10 États par chiffre d'affaires")
plt.xlabel("État")
plt.ylabel("Ventes totales")
plt.tight_layout()
plt.show()

## Quelle est la différence entre New York et la Californie ?

In [ ]:
ny_ca = df[df['State'].isin(['New York', 'California'])]
state_summary = ny_ca.groupby('State')[['Sales', 'Profit']].sum()
print(state_summary)

state_summary.plot(kind='bar', figsize=(8, 5))
plt.title("New York vs California : Sales et Profit")
plt.xlabel("État")
plt.ylabel("Montant total")
plt.tight_layout()
plt.show()

## Qui est un client exceptionnel à New York ?

In [ ]:
clients_ny = (
    df[df['State'] == 'New York']
    .groupby('Customer Name')
    .agg(Sales=('Sales', 'sum'), Profit=('Profit', 'sum'), Orders=('Customer Name', 'count'))
    .sort_values('Profit', ascending=False)
)
print(clients_ny.head(5))

## Existe-t-il des différences de rentabilité entre les États ?

In [ ]:
profit_by_state = df.groupby('State')['Profit'].sum().sort_values(ascending=False)
print(profit_by_state.head(10))

plt.figure(figsize=(10, 5))
profit_by_state.head(10).plot(kind='bar', color='seagreen')
plt.title("Top 10 États par bénéfice total")
plt.xlabel("État")
plt.ylabel("Bénéfice total")
plt.tight_layout()
plt.show()

## Peut-on appliquer le principe de Pareto aux clients et aux bénéfices ?

In [ ]:
client_profit = df.groupby('Customer Name')['Profit'].sum().sort_values(ascending=False)
client_profit_cumsum_pct = client_profit.cumsum() / client_profit.sum() * 100
threshold_index = max(1, int(len(client_profit) * 0.2))
profit_share_top_20 = client_profit.head(threshold_index).sum() / client_profit.sum() * 100

print(f"Les 20% premiers clients génèrent {profit_share_top_20:.2f}% des bénéfices.")
print("Conclusion Pareto :", "Oui" if profit_share_top_20 >= 80 else "Non")

plt.figure(figsize=(10, 5))
plt.plot(range(1, len(client_profit_cumsum_pct) + 1), client_profit_cumsum_pct.values, marker='o', linewidth=1)
plt.axhline(80, color='red', linestyle='--', label='Seuil 80%')
plt.axvline(threshold_index, color='green', linestyle='--', label='20% des clients')
plt.title("Courbe cumulative des bénéfices par client")
plt.xlabel("Nombre de clients classés par bénéfice")
plt.ylabel("Pourcentage cumulé des bénéfices")
plt.legend()
plt.tight_layout()
plt.show()

## Quelles sont les 20 premières villes en termes de chiffre d'affaires et de bénéfice ?

In [ ]:
city_summary = df.groupby('City')[['Sales', 'Profit']].sum()
top_20_cities_sales = city_summary.sort_values('Sales', ascending=False).head(20)
top_20_cities_profit = city_summary.sort_values('Profit', ascending=False).head(20)

print("Top 20 villes par chiffre d'affaires :")
print(top_20_cities_sales)

print("\nTop 20 villes par bénéfice :")
print(top_20_cities_profit)

plt.figure(figsize=(12, 6))
top_20_cities_sales['Sales'].sort_values().plot(kind='barh', color='royalblue')
plt.title("Top 20 villes par chiffre d'affaires")
plt.xlabel("Ventes totales")
plt.ylabel("Ville")
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 6))
top_20_cities_profit['Profit'].sort_values().plot(kind='barh', color='darkorange')
plt.title("Top 20 villes par bénéfice")
plt.xlabel("Bénéfice total")
plt.ylabel("Ville")
plt.tight_layout()
plt.show()

## Quels sont les 20 meilleurs clients en termes de ventes ?

In [ ]:
client_sales = df.groupby('Customer Name')['Sales'].sum().sort_values(ascending=False)
print("Top 20 clients en termes de ventes :")
print(client_sales.head(20))

plt.figure(figsize=(12, 6))
client_sales.head(20).sort_values().plot(kind='barh', color='mediumpurple')
plt.title("Top 20 clients par ventes")
plt.xlabel("Ventes totales")
plt.ylabel("Client")
plt.tight_layout()
plt.show()

sales_cumsum_pct = client_sales.cumsum() / client_sales.sum() * 100
threshold_index_sales = max(1, int(len(client_sales) * 0.2))
sales_share_top_20 = client_sales.head(threshold_index_sales).sum() / client_sales.sum() * 100

print(f"Les 20% premiers clients génèrent {sales_share_top_20:.2f}% des ventes.")
print("Conclusion Pareto ventes :", "Oui" if sales_share_top_20 >= 80 else "Non")

plt.figure(figsize=(10, 5))
plt.plot(range(1, len(sales_cumsum_pct) + 1), sales_cumsum_pct.values, color='black')
plt.axhline(80, color='red', linestyle='--', label='Seuil 80%')
plt.axvline(threshold_index_sales, color='green', linestyle='--', label='20% des clients')
plt.title("Courbe cumulative des ventes par client")
plt.xlabel("Nombre de clients classés par ventes")
plt.ylabel("Pourcentage cumulé des ventes")
plt.legend()
plt.tight_layout()
plt.show()

## Conclusion et recommandations marketing

Les analyses montrent qu'il faut privilégier les zones où les profits sont les plus élevés, en particulier les États et villes qui apparaissent dans les premiers rangs des classements de ventes et de bénéfices. Les clients qui génèrent une part importante du bénéfice total doivent être ciblés en priorité, car le principe de Pareto est vérifié pour les bénéfices mais pas pour les ventes. Pour améliorer la performance marketing, il est donc pertinent de concentrer les actions sur les villes rentables, de fidéliser les meilleurs clients et d'éviter de disperser les efforts sur les marchés peu profitables.